# Multi-Regional AeroMAPS Tutorial — EU Domestic & International

This notebook is a minimal introduction to multi-regional modeling in AeroMAPS.
It runs two regions in parallel — **EU Domestic** (`EU_DOM`) and **EU International** (`EU_INT`) —
and shows how to create, run, and visualize a multi-regional process.

The same pattern scales to any number of regions (see `examples_multi_regional_simple.ipynb`).

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from aeromaps import create_multi_regional_process
from aeromaps.utils.functions import create_partitioning

figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)

%matplotlib widget

## 1. Prepare Region Data

`create_partitioning` reads the AeroSCOPE CSV for each region and generates
the `partitioning_updated_inputs.json` file that initialises AeroMAPS traffic data.
Run this once (or whenever the CSV data changes).

In [ ]:
region_folders = ["region_eu_dom", "region_eu_int"]
data_path = Path("data")

for folder_name in region_folders:
    folder_path = data_path / folder_name
    csv_file = folder_path / "dataframe_aeromaps.csv"
    create_partitioning(file=str(csv_file), path=str(folder_path))
    print(f"✓ Prepared: {folder_name}")

## 2. Create and Run the Multi-Regional Process

The regionalisation configuration file (`data/regionalisation_europe.yaml`) lists
the two regions, their individual config files, and the variables to aggregate globally.

In [ ]:
process = create_multi_regional_process(
    configuration_file="regionalisation_europe.yaml",
    disable_execution_statistics=True,
)

print(f"Mode     : {process.execution_mode}")
print(f"Regions  : {process.list_regions()}")

process.compute(parallel=False)
print("\n✓ Computation complete")

## 3. Explore Outputs

All outputs are stored in `process.data["vector_outputs"]` with region-prefixed column names
(e.g. `EU_DOM:co2_emissions_including_energy`) plus aggregated `overall:` columns.

In [ ]:
vo = process.data["vector_outputs"]

print("Available regions:", process.list_regions())
print("\nCO₂ emissions incl. energy (Mt) at key years:")
cols = [
    "EU_DOM:co2_emissions_including_energy",
    "EU_INT:co2_emissions_including_energy",
    "overall:co2_emissions_including_energy",
]
print(vo[cols].loc[[2019, 2030, 2050]].round(1).to_string())

## 4. Visualize — CO₂ Emissions by Region

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

series = [
    ("EU_DOM:co2_emissions_including_energy", "EU Domestic", "#4daf4a", "--"),
    ("EU_INT:co2_emissions_including_energy", "EU International", "#2171b5", ":"),
    ("overall:co2_emissions_including_energy", "EU DOM + INT combined", "black", "-"),
]

for col, label, color, ls in series:
    if col in vo.columns:
        vo[col].plot(ax=ax, label=label, color=color, linestyle=ls, linewidth=1.8)

ax.set_xlabel("Year")
ax.set_ylabel("CO₂ Emissions (Mt)")
ax.set_xlim(2019, 2050)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / "eu_co2_emissions.pdf", format="pdf", bbox_inches="tight")
plt.show()

## 5. Visualize — Emissions Decomposition (Waterfall)

Fill-between areas show the contribution of each mitigation lever
(fleet renewal, future aircraft, operations, low-carbon energy, CORSIA offsets).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

fills = [
    (
        "co2_emissions_2019technology",
        "co2_emissions_including_aircraft_efficiency",
        "#FFE082",
        0.55,
        "Fleet / efficiency",
    ),
    (
        "co2_emissions_including_aircraft_efficiency",
        "co2_emissions_including_load_factor",
        "#FFC107",
        0.55,
        "Operations",
    ),
    (
        "co2_emissions_including_load_factor",
        "co2_emissions_including_energy",
        "#66BB6A",
        0.50,
        "Low-carbon energy",
    ),
]

for ax, region_id in zip(axes, ["EU_DOM", "EU_INT"]):
    years = vo.index

    s_bau = vo.get(f"{region_id}:co2_emissions_2019technology")
    net_col = f"{region_id}:co2_emissions_including_energy"
    off_col = f"{region_id}:carbon_offset"
    s_net = (vo[net_col] - vo[off_col]) if net_col in vo and off_col in vo else None

    if s_bau is not None:
        ax.plot(years, s_bau, color="#E0A800", linewidth=1.0, alpha=0.6)
    if s_net is not None:
        ax.plot(years, s_net, color="#757575", linewidth=1.2)

    for top_key, bottom_key, color, alpha, label in fills:
        top = vo.get(f"{region_id}:{top_key}")
        bottom = vo.get(f"{region_id}:{bottom_key}")
        if top is not None and bottom is not None:
            ax.fill_between(
                years,
                top,
                bottom,
                color=color,
                alpha=alpha,
                label=label,
                edgecolor=color,
                linewidth=0.3,
            )

    ax.set_title(f"{region_id.replace('_', ' ')}", fontsize=11)
    ax.set_xlabel("Year")
    ax.set_ylabel("CO₂ Emissions (Mt)")
    ax.set_xlim(2019, 2050)
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.4, linewidth=0.8)
    ax.grid(True, alpha=0.25)

axes[0].legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.savefig(figures_dir / "eu_emissions_waterfall.pdf", format="pdf", bbox_inches="tight")
plt.show()